In [1]:
import pandas as pd
import seaborn as sns 
import matplotlib.pyplot as plt 
import numpy as np 

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.pipeline import Pipeline

from sklearn.metrics import make_scorer, mean_absolute_error, root_mean_squared_error, r2_score


In [2]:
df = pd.read_csv("modelling (1).csv")

In [3]:
df.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,...,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,Year,Month,Day,WeekOfYear
0,1097,2,2013-01-01,5961,1405,1,0,1,1,1,...,3.0,2002.0,0,0.0,0.0,NaN,2013,1,1,1
1,85,2,2013-01-01,4220,619,1,0,1,1,1,...,10.0,2011.0,0,0.0,0.0,NaN,2013,1,1,1
2,259,2,2013-01-01,6851,1444,1,0,1,1,1,...,0.0,0.0,0,0.0,0.0,NaN,2013,1,1,1
3,262,2,2013-01-01,17267,2875,1,0,1,1,1,...,5.0,2013.0,0,0.0,0.0,NaN,2013,1,1,1
4,335,2,2013-01-01,2401,482,1,0,1,1,1,...,0.0,0.0,1,31.0,2013.0,"Jan,Apr,Jul,Oct",2013,1,1,1


In [4]:
df.columns.to_list()

['Store',
 'DayOfWeek',
 'Date',
 'Sales',
 'Customers',
 'Open',
 'Promo',
 'StateHoliday',
 'SchoolHoliday',
 'StoreType',
 'Assortment',
 'CompetitionDistance',
 'CompetitionOpenSinceMonth',
 'CompetitionOpenSinceYear',
 'Promo2',
 'Promo2SinceWeek',
 'Promo2SinceYear',
 'PromoInterval',
 'Year',
 'Month',
 'Day',
 'WeekOfYear']

In [5]:
df['Assortment'].unique()

array([1, 0, 2])

In [6]:

features = [
    'DayOfWeek',
    'Open',
    'Promo',
    'StateHoliday',
    'SchoolHoliday',
    'StoreType',
    'Assortment',
    'CompetitionDistance',
    'CompetitionOpenSinceMonth',
    'CompetitionOpenSinceYear',
    'Promo2',
    'Promo2SinceWeek',
    'Promo2SinceYear',
    'Year',
    'Month',
    'Day',
    'WeekOfYear'
]

X = df[features]
y = df['Sales']

In [7]:
numerical_features = [
    'DayOfWeek',
    'CompetitionDistance',
    'CompetitionOpenSinceMonth',
    'CompetitionOpenSinceYear',
    'Promo2SinceWeek',
    'Promo2SinceYear',
    'Year',
    'Month',
    'Day',
    'WeekOfYear'
]


In [8]:
categorical_features = [
    'StateHoliday',
    'StoreType',
    'Assortment'
]

In [9]:
binary_features = [
    'Open',
    'Promo',
    'SchoolHoliday',
    'Promo2'
]


In [10]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough'
)


In [11]:
mlp = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    max_iter=2000,
    random_state=0
)

In [12]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('mlp', mlp)
])
tscv = TimeSeriesSplit(n_splits=5)

In [13]:
scoring = {
    'MAE': 'neg_mean_absolute_error',
    'RMSE': 'neg_root_mean_squared_error',
    'R2': 'r2'
}

In [ ]:
results = cross_validate(
    pipeline,
    X,
    y,
    cv=tscv,
    scoring=scoring,
    return_train_score=True
)

In [ ]:
mae_scores = -results['test_MAE']
rmse_scores = -results['test_RMSE']
r2_scores = results['test_R2']


In [ ]:
print("MAE scores:", mae_scores)
print("Mean MAE:", mae_scores.mean())
print("Standard deviation MAE:", mae_scores.std())

print("\nRMSE scores:", rmse_scores)
print("Mean RMSE:", rmse_scores.mean())
print("Standard deviation RMSE:", rmse_scores.std())

print("\nR² scores:", r2_scores)
print("Mean R²:", r2_scores.mean())
print("Standard deviation R²:", r2_scores.std())